# Functions and DataFrames

In [1]:
import polars as pl
import matplotlib
path_data = '../../assets/data/'
matplotlib.use('Agg')
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

We are building up a useful inventory of techniques for identifying patterns and themes in a data set by using functions already available in Python. We will now explore a core feature of the Python programming language: function definition.

We have used functions extensively already in this text, but never defined a function of our own. The purpose of defining a function is to give a name to a computational process that may be applied multiple times. There are many situations in computing that require repeated computation. For example, it is often the case that we want to perform the same manipulation on every value in a column of a DataFrame.

**Defining a Function**

The definition of the `double` function below simply doubles a number.

In [2]:
# Our first function definition

def double(x):
    """ Double x """
    return 2*x

We start any function definition by writing `def`.  Here is a breakdown of the other parts (the *syntax*) of this small function:

![The first line of the function is "def double(x)". "double(x)" is annotated as the signature. "double" is the name of the function and "x" is the first (and only) argument. The next line is indented and shows three double quotes surrounding "Double x". This is annotated as the documentation or docstring that describes what the function does. The following line is "return 2*x" and annotated as the body. The body is all the code that runs each time you call the function. Return tells Python what the value of each call to the function is: it's the value of the expression after return. Finally, the indendation of the docstring and the body is highlighted. This indentation tells Python that those lines are part of the body.](../../images/function_definition.jpg)

When we run the cell above, no particular number is doubled, and the code inside the body of `double` is not yet evaluated.  In this respect, our function is analogous to a *recipe*.  Each time we follow the instructions in a recipe, we need to start with ingredients.  Each time we want to use our function to double a number, we need to specify a number.

We can call `double` in exactly the same way we have called other functions. Each time we do that, the code in the body is executed, with the value of the argument given the name `x`.

In [3]:
double(17)

34


In [4]:
double(-0.6/4)

-0.3


The two expressions above are both *call expressions*. In the second one, the value of the expression `-0.6/4` is computed and then passed as the argument named `x` to the `double` function. Each call expresson results in the body of `double` being executed, but with a different value of `x`.

The body of `double` has only a single line:

`return 2*x`

Executing this *`return` statement* completes execution of the `double` function's body and computes the value of the call expression.

The argument to `double` can be any expression, as long as its value is a number.  For example, it can be a name.  The `double` function does not know or care how its argument is computed or stored; its only job is to execute its own body using the values of the arguments passed to it.

In [5]:
any_name = 42
double(any_name)

84


The argument can also be any value that can be doubled. For example, a whole Polars Series of numbers can be passed as an argument to `double`, and the result will be another Polars Series.

In [6]:
double(pl.Series("numbers", [3, 4, 5]))

numbers
i64
6
8
10


However, names that are defined inside a function, including arguments like `double`'s `x`, have only a fleeting existence.  They are defined only while the function is being called, and they are only accessible inside the body of the function. We can't refer to `x` outside the body of `double`. The technical terminology is that `x` has *local scope*.

Therefore the name `x` isn't recognized outside the body of the function, even though we have called `double` in the cells above.

In [7]:
x

NameError: name 'x' is not defined

**Docstrings.** Though `double` is relatively easy to understand, many functions perform complicated tasks and are difficult to use without explanation. (You may have discovered this yourself!) Therefore, a well-composed function has a name that evokes its behavior, as well as documentation.  In Python, this is called a *docstring* — a description of its behavior and expectations about its arguments. The docstring can also show example calls to the function, where the call is preceded by `>>>`.

A docstring can be any string, as long as it is the first thing in a function's body. Docstrings are typically defined using triple quotation marks at the start and end, which allows a string to span multiple lines. The first line is conventionally a complete but short description of the function, while following lines provide further guidance to future users of the function.

Here is a definition of a function called `percent` that takes two arguments. The definition includes a docstring.

In [8]:
# A function with more than one argument

def percent(x, total):
    """Convert x to a percentage of total.
    
    More precisely, this function divides x by total,
    multiplies the result by 100, and rounds the result
    to two decimal places.
    
    >>> percent(4, 16)
    25.0
    >>> percent(1, 6)
    16.67
    """
    return round((x/total)*100, 2)

In [9]:
percent(33, 200)

16.5


Contrast the function `percent` defined above with the function `percents` defined below. The latter takes a Polars Series as its argument, and converts all the numbers in the Series to percents out of the total of the values in the Series. The percents are all rounded to two decimal places, this time replacing `round` by `Series.round` because the argument is a Polars Series and not a number.

In [10]:
def percents(counts):
    """Convert the values in array_x to percents out of the total of array_x."""
    total = counts.sum()
    return ((counts / total) * 100).round(2)

The function `percents` returns a Polars Series of percents that add up to 100 apart from rounding.

In [11]:
some_array = pl.Series("some_array", [7, 10, 4])
percents(some_array)

some_array
f64
33.33
47.62
19.05


It is helpful to understand the steps Python takes to execute a function.  To facilitate this, we have put a function definition and a call to that function in the same cell below.

In [12]:
def biggest_difference(array_x):
    """Find the biggest difference in absolute value between two adjacent elements of array_x."""
    # POLARS_CHANGE: diff keeps a leading null. Drop it so the Series has the same length as np.diff.
    diffs = array_x.diff(null_behavior="drop")
    absolute_diffs = abs(diffs)
    return max(absolute_diffs)

some_numbers = pl.Series("some_numbers", [2, 4, 5, 6, 4, -1, 1])
big_diff = biggest_difference(some_numbers)
print("The biggest difference is", big_diff)

The biggest difference is 5


Here is what happens when we run that cell:

![The overall layout consists of the same code snippet repeated on the left, descriptive text boxes with callout lines pointing to specific parts of the code, and a "Names defined in scope" section on the right, which shows the state of variables at different stages. The code snippet starts by defining the biggest_difference function, which takes array_x as input and has the following docstring: ‘Find the biggest difference in absolute value between two adjacent elements of array_x.’ The first line of code annotation is ‘The function biggest_difference is defined.’ The next repetition of the code annotates the line following the end of the function with’ The array some_numbers is defined.’ and under ‘Names defined in scope’ now is ‘biggest_difference = <a function>’. The next annotation is for the next line ‘big_diff = biggest_difference(some_numbers)’ with ‘Our function is called. Before this line finishes, Python executes its body.’ some_numbers has been added to ’Names defined in scope.’ The next annotation goes back to the function body and states ‘The argument is given name array_x. The function’s first line (docstring) does nothing.’ And we see array_x now has the value of some_numbers. The next line of code after the docstring is ‘diffs = np.diff(array_x)’ with annotation ‘The array diffs is defined.’ The next line of code is ‘absolute_diffs = abs(diffs)’ with annotation ‘The array absolute_diffs is defined.’ and we see ‘diffs’ in the ‘Names defined in scope’ with the calculated values filled in the array. The next line of code is ’return max(absolute_diffs)’ with annotation ‘The value of max(absolute_diffs) is computed and becomes the value of the call biggest_difference(some_numbers)’ and ‘absolute_diffs’ added to ‘Names defined in scope.’ Finally, the annotation jumps back to the line after the function call with the annotation ‘The function call is done, so array_x, diffs, and absolute_diffs disappear. some_numbers reappears, and big_diff is defined as the value of the call. Finally the print statement happens.’](../../images/function_execution.jpg)

**Multiple Arguments**

There can be multiple ways to generalize an expression or block of code, and so a function can take multiple arguments that each determine different aspects of the result. For example, the `percents` function we defined previously rounded to two decimal places every time. The following two-argument definition allows different calls to round to different amounts.

In [13]:
def percents(counts, decimal_places):
    """Convert the values in array_x to percents out of the total of array_x."""
    total = counts.sum()
    return ((counts / total) * 100).round(decimal_places)

parts = pl.Series("parts", [2, 1, 4])
print("Rounded to 1 decimal place: ", percents(parts, 1))
print("Rounded to 2 decimal places:", percents(parts, 2))
print("Rounded to 3 decimal places:", percents(parts, 3))

Rounded to 1 decimal place:  shape: (3,)
Series: 'parts' [f64]
[
	28.6
	14.3
	57.1
]
Rounded to 2 decimal places: shape: (3,)
Series: 'parts' [f64]
[
	28.57
	14.29
	57.14
]
Rounded to 3 decimal places: shape: (3,)
Series: 'parts' [f64]
[
	28.571
	14.286
	57.143
]


The flexibility of this new definition comes at a small price: each time the function is called, the number of decimal places must be specified. Default argument values allow a function to be called with a variable number of arguments; any argument that isn't specified in the call expression is given its default value, which is stated in the first line of the `def` statement. For example, in this final definition of `percents`, the optional argument `decimal_places` is given a default value of 2.

In [14]:
def percents(counts, decimal_places=2):
    """Convert the values in array_x to percents out of the total of array_x."""
    total = counts.sum()
    return ((counts / total) * 100).round(decimal_places)

parts = pl.Series("parts", [2, 1, 4])
print("Rounded to 1 decimal place:", percents(parts, 1))
print("Rounded to the default number of decimal places:", percents(parts))

Rounded to 1 decimal place: shape: (3,)
Series: 'parts' [f64]
[
	28.6
	14.3
	57.1
]
Rounded to the default number of decimal places: shape: (3,)
Series: 'parts' [f64]
[
	28.57
	14.29
	57.14
]


**Note: Methods**

Functions are called by placing argument expressions in parentheses after the function name. Any function that is defined in isolation is called in this way. You have also seen examples of methods, which are like functions but are called using dot notation, such as `some_dataframe.sort(some_label)`. The functions that you define will always be called using the function name first, passing in all of the arguments.